# Installing Python, IDEs, and the REPL

**Topic:** `1.2` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Format an environment report line

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** f-strings, tuples, sys

**Explanation and reasoning**

Slicing to two components drops the micro version so (3, 12, 1) is treated the same as (3, 12). The verdict comes from a tuple comparison, which orders numerically, and the rendered string is built separately so display and comparison cannot drift apart.

In [ ]:
MINIMUM = (3, 12)


def report_line(version_tuple) -> str:
    version = '.'.join(str(part) for part in version_tuple[:2])
    verdict = 'OK' if tuple(version_tuple[:2]) >= MINIMUM else 'TOO OLD'
    return f'python {version} {verdict}'

**Complexity**

Time O(k) in the number of components; space O(1).

**Edge cases and failure modes**

A one-element tuple renders as a single number and compares as expected.

**Alternative approaches**

An f-string with the tuple unpacked inline is shorter but harder to read.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert report_line((3, 12)) == 'python 3.12 OK'
assert report_line((3, 9)) == 'python 3.9 TOO OLD'
assert report_line((4, 0)) == 'python 4.0 OK'

---

## Solution 2 — Tuple-safe version comparison

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** tuples, comparison, validation

**Explanation and reasoning**

Normalising both arguments in one helper guarantees the comparison is numeric on both sides. Slicing to two components means '3.12.1' and '3.12' agree, and raising TypeError for anything else keeps a silent misinterpretation impossible.

In [ ]:
def _as_tuple(version):
    if isinstance(version, str):
        return tuple(int(part) for part in version.split('.'))
    if isinstance(version, (tuple, list)):
        return tuple(int(part) for part in version)
    raise TypeError(f'unsupported version: {version!r}')


def version_ok(running, minimum) -> bool:
    return _as_tuple(running)[:2] >= _as_tuple(minimum)[:2]

**Complexity**

Time O(k) in the number of components; space O(k).

**Edge cases and failure modes**

An empty string raises ValueError from int(''), which is the correct signal.

**Alternative approaches**

packaging.version handles pre-release suffixes that this simple form ignores.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert version_ok('3.12', (3, 12)) is True
assert version_ok('3.9', '3.12') is False
assert version_ok((3, 12, 1), (3, 12)) is True

---

## Solution 3 — Describe the running interpreter

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** sys, dicts, modules

**Explanation and reasoning**

Each field is read directly from the runtime rather than recomputed. hasattr(sys, 'ps1') is the documented way to detect an interactive session, and splitting sys.version gives the short form without the build metadata.

In [ ]:
import platform
import sys


def describe_environment() -> dict:
    return {
        'executable': sys.executable,
        'version': sys.version.split()[0],
        'platform': f'{platform.system()} {platform.machine()}',
        'interactive': hasattr(sys, 'ps1'),
    }

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

An embedded interpreter may report an empty executable path; the key is still present.

**Alternative approaches**

platform.uname() gives richer detail but returns a named tuple.

**Tests — run the cell to verify the reference solution**

In [ ]:
d = describe_environment()
assert set(d) == {'executable', 'version', 'platform', 'interactive'}
assert isinstance(d['interactive'], bool)
assert d['version'].count('.') >= 1

---

## Solution 4 — Check an optional dependency safely

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** importlib, modules, booleans

**Explanation and reasoning**

find_spec answers whether an import would resolve without executing the module, which is the behaviour the task requires. A parent package that fails during initialisation raises ImportError or ValueError rather than returning None, so those are caught and reported as unavailable.

In [ ]:
import importlib.util


def is_available(name) -> bool:
    if not isinstance(name, str):
        raise TypeError(f'module name must be a string, got {type(name).__name__}')
    if not name:
        return False
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False

**Complexity**

Time O(n) in sys.path length; space O(1).

**Edge cases and failure modes**

An empty name returns False; a dotted name such as 'os.path' resolves.

**Alternative approaches**

A guarded import inside try/except is simpler but executes the module's code.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert is_available('sqlite3') is True
assert is_available('definitely_not_a_module_xyz') is False

---

## Solution 5 — Join path segments portably

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** pathlib, strings, composition

**Explanation and reasoning**

The pathlib operator performs the platform-specific joining and normalisation, so a doubled separator cannot be produced on any system. Falsy and whitespace-only segments are skipped rather than appended, which keeps a stray empty argument from creating a confusing path.

In [ ]:
from pathlib import Path


def build_path(base, *parts) -> Path:
    path = Path(base)
    for part in parts:
        if part and part.strip():
            path = path / part.strip()
    return path

**Complexity**

Time O(n) in the number of segments; space O(n) in path length.

**Edge cases and failure modes**

A base with a trailing slash is normalised; no segments returns the base Path.

**Alternative approaches**

pathlib.PurePath handles the same logic for virtual filesystems.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert str(build_path('/opt/robo', 'venvs', 'main')) == '/opt/robo/venvs/main'
assert str(build_path('/opt/robo', '', '  ', 'main')) == '/opt/robo/main'

---

## Solution 6 — Parse a pinned dependency block

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, dicts, parsing

**Explanation and reasoning**

Semicolons are normalised to newlines first so one parser handles both layouts. The for/else construct raises only when no operator was found in a line, which is exactly the malformed case, and stripping comments before parsing means a trailing comment never splits a constraint.

In [ ]:
OPERATORS = ('==', '>=', '<=', '~=', '!=', '>', '<')


def parse_pins(text: str) -> dict:
    pins: dict = {}
    for raw in text.replace(';', '\n').splitlines():
        line = raw.split('#', 1)[0].strip()
        if not line:
            continue
        for op in OPERATORS:
            if op in line:
                name, _, constraint = line.partition(op)
                name = name.strip()
                if not name:
                    raise ValueError(f'missing package name: {raw!r}')
                pins[name] = op + constraint.strip()
                break
        else:
            raise ValueError(f'no version operator in {raw!r}')
    return pins

**Complexity**

Time O(n) in the input length; space O(k) for k packages.

**Edge cases and failure modes**

Blank and comment-only lines are skipped; 'numpy' with no operator raises ValueError.

**Alternative approaches**

A regular expression with a named group is shorter but less explicit.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert parse_pins('numpy==2.1.0; requests>=2.31') == {'numpy': '==2.1.0', 'requests': '>=2.31'}
assert parse_pins('# comment\nnumpy==2.1.0') == {'numpy': '==2.1.0'}

---

## Solution 7 — Verify required modules against what is present

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** sets, dicts, validation

**Explanation and reasoning**

Set difference removes duplicates on both sides and computes the missing names in one operation. Sorting makes the result deterministic, which matters because this list is printed in reports and compared in tests.

In [ ]:
def verify_modules(required, available) -> tuple[bool, list[str]]:
    missing = sorted(set(required) - set(available))
    return (not missing), missing

**Complexity**

Time O(n + m) in the input sizes; space O(n).

**Edge cases and failure modes**

An empty `required` yields an empty list and therefore True.

**Alternative approaches**

A comprehension over required with a membership test is equivalent.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert verify_modules(['json', 'requests'], ['json']) == (False, ['requests'])
assert verify_modules([], []) == (True, [])

---

## Solution 8 — Resolve the first available interpreter launcher

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** shutil.which, loops, error reporting

**Explanation and reasoning**

which() implements the same PATH scan the operating system uses, so the result matches what actually happens when the launcher is typed. Building the tried list as we go means the error names every candidate, which is the information needed to diagnose a broken PATH.

In [ ]:
import shutil


def pick_interpreter(candidates) -> str:
    tried = []
    for name in candidates:
        tried.append(name)
        found = shutil.which(name)
        if found:
            return found
    raise FileNotFoundError(
        'no interpreter found; tried: ' + ', '.join(tried)
    )

**Complexity**

Time O(n * p) for n candidates over p PATH entries; space O(n).

**Edge cases and failure modes**

A non-executable file with a matching name is skipped by which().

**Alternative approaches**

shutil.which on a single name is simpler but cannot express a preference order.

**Tests — run the cell to verify the reference solution**

In [ ]:
import shutil
name = pick_interpreter(['python3.12', 'python3'])
assert name == shutil.which(name) or name == shutil.which('python3.12')

---

## Solution 9 — Summarise the module search path

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** sys.path, sets, ordering

**Explanation and reasoning**

A single pass with a seen-set gives the unique count, while a separate list preserves the first-seen order of repeated entries. Checking membership in the duplicates list as well keeps that list unique without a second set.

In [ ]:
def summarise_search_path(entries) -> dict:
    entries = list(entries)
    seen: set = set()
    duplicates: list[str] = []
    for entry in entries:
        if entry in seen:
            if entry not in duplicates:
                duplicates.append(entry)
        else:
            seen.add(entry)
    return {'total': len(entries), 'unique': len(seen), 'duplicates': duplicates}

**Complexity**

Time O(n) in the number of entries; space O(n).

**Edge cases and failure modes**

An empty input returns zero counts and an empty duplicate list.

**Alternative approaches**

Counter preserves insertion order in modern Python and is shorter.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert summarise_search_path(['a', 'b', 'a', 'c', 'b']) == {'total': 5, 'unique': 3, 'duplicates': ['a', 'b']}
assert summarise_search_path([]) == {'total': 0, 'unique': 0, 'duplicates': []}

---

## Solution 10 — Assemble a machine pre-flight report

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, composition, validation, sys

**Explanation and reasoning**

The function collects problems instead of raising, because a check that reports every fault at once is far more useful than one that stops at the first. ready is then derived from the emptiness of problems, which makes it impossible for the flag to disagree with the list it summarises.

In [ ]:
import importlib.util
import sys

DEFAULT_MINIMUM = (3, 12)
DEFAULT_REQUIRED = ('json', 'sqlite3')


def preflight_report(minimum=DEFAULT_MINIMUM, required=DEFAULT_REQUIRED) -> dict:
    problems: list[str] = []
    running = tuple(sys.version_info[:2])
    version = sys.version.split()[0]
    if running < tuple(minimum[:2]):
        problems.append(f'version {version} older than {minimum}')
    for name in required:
        try:
            present = importlib.util.find_spec(name) is not None
        except (ImportError, ValueError):
            present = False
        if not present:
            problems.append(f'missing module: {name}')
    return {'ready': not problems, 'version': version, 'problems': problems}

**Complexity**

Time O(m) for m required modules; space O(m).

**Edge cases and failure modes**

An old interpreter still returns a report with ready False.

**Alternative approaches**

A dataclass result type would be clearer in larger codebases.

**Tests — run the cell to verify the reference solution**

In [ ]:
report = preflight_report()
assert report['ready'] is True
assert report['problems'] == []
assert report['version'].count('.') == 2

---

# Robotics challenge solution — ROBO-X Challenge: Bring-Up Readiness Gate

**Explanation**

Combining the environment facts with a live simulator read shows both halves of pre-flight: the machine is new enough, and the subsystems it must drive actually answer. Each channel is isolated so one failing sensor degrades the report instead of destroying it.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

REQUIRED_CHANNELS = ('distance', 'temperature', 'battery_voltage')


def environment_readiness(channels=None) -> dict:
    wanted = list(channels) if channels else list(REQUIRED_CHANNELS)
    robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
    readings, failed = {}, []
    for channel in wanted:
        try:
            readings[channel] = robot.read_sensor(channel)
        except Exception:  # SensorError, or an unknown channel name
            readings[channel] = None
            failed.append(channel)
    state = robot.status()
    return {
        'ready': not failed,
        'version': sys.version.split()[0],
        'battery_pct': state['battery_pct'],
        'readings': readings,
        'failed_channels': failed,
    }

**Complexity**

Time O(c) for c channels; space O(c).

**Notes and trade-offs**

An unknown channel name is recorded as failed rather than raising.

In [ ]:
report = environment_readiness()
assert report['ready'] is True
assert report['failed_channels'] == []
assert report['battery_pct'] > 0

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.